In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import celldega as dega
import numpy as np
import pandas as pd
from anndata import AnnData
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import scanpy as sc
from spatialdata_io import xenium
import spatialdata as sd
import os
from scipy.sparse import csr_matrix
print(dega.__version__)

objc[36486]: Class GNotificationCenterDelegate is implemented in both /opt/homebrew/Cellar/glib/2.84.3/lib/libgio-2.0.0.dylib (0x30bfb84b8) and /Users/jishar/anaconda3/lib/libgio-2.0.0.dylib (0x30b95e310). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


0.13.0a9


In [3]:
from ipywidgets import Widget
Widget.close_all()

## Real Data

In [4]:
data_dir = "data/xenium_data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs"

## Make AnnData

In [5]:
# # Ingest xenium data raw output folder using spatialdata-io
# sdata = xenium(data_dir)

# # Write sdata to a zarr file
# zarr_path = f"{data_dir}.zarr"

# # Check if the zarr file already exists
# if os.path.exists(zarr_path):
#     print(f"The file {zarr_path} already exists.")
# else:
#     # If the file does not exist, write the data
#     sdata.write(zarr_path)
#     print(f"Data written to {zarr_path} successfully.")

# # Read zarr file using spatialdata
# sdata = sd.read_zarr(zarr_path)

# # Create anndata from sdata.tables['table]
# adata = sdata.tables["table"]
# adata.write_h5ad(f'{data_dir}.h5ad')

In [6]:
# # Load h5ad file
# adata = sc.read_h5ad(f'{data_dir}.h5ad')
# adata.obs.set_index('cell_id', inplace=True)
# adata

## Scanpy processing

In [7]:
# sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
# sc.pp.filter_cells(adata, min_counts=10)
# sc.pp.filter_genes(adata, min_cells=5)

# adata.X = csr_matrix(adata.X)

# adata.layers["counts"] = adata.X

# sc.pp.normalize_total(adata, inplace=True)

# sc.pp.log1p(adata)

# sc.pp.highly_variable_genes(adata, n_top_genes=2000)
# adata = adata[:, adata.var.highly_variable].copy()

# sc.pp.pca(adata)

# sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=10)

# sc.tl.leiden(adata, resolution=1.0)

In [8]:
# adata.write_h5ad(f'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_sc_processed.h5ad')

In [9]:
adata = sc.read_h5ad(f'data/xenium_data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_sc_processed.h5ad')

In [10]:
# Convert the results to a pandas DataFrame
def get_ranked_genes_df(adata, n_genes=100):
    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    dfs = []
    for group in groups:
        df = pd.DataFrame({
            'gene': result['names'][group][:n_genes],
            'logfoldchanges': result['logfoldchanges'][group][:n_genes],
            'pvals': result['pvals'][group][:n_genes],
            'pvals_adj': result['pvals_adj'][group][:n_genes],
            'scores': result['scores'][group][:n_genes],
            'cluster': group
        })
        dfs.append(df)
    return pd.concat(dfs)

## Rank and save marker genes

In [11]:
# # Run ranking (faster)
# sc.tl.rank_genes_groups(adata, groupby="leiden", method="t-test", use_raw=False, show_progress=True)

# # Save markers
# marker_df = get_ranked_genes_df(adata, n_genes=100)
# marker_df.to_csv("data/xenium_data/marker_genes_by_cluster.csv", index=False)

#### 1. Uploaded "marker_genes_by_cluster.csv" on ChatGPT, and asked for tentative cell types.
#### 2. "Predicted_Cell_Types_by_Cluster.csv" has the predicted cell types for each cluster based on the top 10 marker genes, with the cluster number included in the label.

In [12]:
pred_cell_types_df = pd.read_csv("data/xenium_data/Predicted_Cell_Types_by_Cluster.csv")
pred_cell_types_df.drop(['Unnamed: 0'], axis=1, inplace=True)
pred_cell_types_df["category"] = pred_cell_types_df["predicted_cell_type"].str.split("_").str[0]
pred_cell_types_df[:5]

,cluster,predicted_cell_type,category
0,0,Unknown_0,Unknown
1,1,Epithelial_1,Epithelial
2,2,Smooth muscle_2,Smooth muscle
3,3,Fibroblast_3,Fibroblast
4,4,Epithelial_4,Epithelial


### Make hextiles

In [13]:
data = dega.nbhd._get_gdf_cell(adata)
gdf_nbhd = dega.nbhd.generate_hex_grid(data, diameter=150)

In [14]:
adata_nbp, gdf_nbhd = dega.nbhd.calc_nbp(data, gdf_nbhd, category="cluster")

Calculating NBP


## Cell-cluster by Hextile using NBHD module methods

In [15]:
# Clustering
sc.pp.normalize_total(adata_nbp, inplace=True)
sc.pp.log1p(adata_nbp)
sc.pp.neighbors(adata_nbp, n_neighbors=10)
sc.tl.leiden(adata_nbp, resolution=1)

In [16]:
population_distribution = pd.DataFrame(
    adata_nbp.X, index=adata_nbp.obs_names, columns=adata_nbp.var_names
)

In [17]:
# Add clustering and proportions to hex GeoDataFrame
gdf_nbhd = gdf_nbhd.set_index("name")
gdf_nbhd["leiden"] = adata_nbp.obs["leiden"].values
gdf_nbhd["niche"] = [f"{cluster}" for cluster in adata_nbp.obs["leiden"].values]
gdf_nbhd = gdf_nbhd.join(population_distribution)
gdf_nbhd.reset_index(inplace=True)

In [18]:
# Dissolve to form niches
gdf_niche = dega.nbhd._dissolve_by_category(gdf_nbhd, "leiden")
gdf_niche["name"] = [f"{c}" for c in gdf_niche["leiden"]]

## SKIP - Clustergram: hextile_nbhd-by-cell_population

In [19]:
# gdf_nbhd_ = gdf_nbhd.drop(['geometry', 'leiden', 'niche'], axis=1)
# gdf_nbhd_.set_index('name', inplace=True)
# gdf_nbhd_ = gdf_nbhd_[(gdf_nbhd_ != 0).any(axis=1)]
# gdf_nbhd_ = gdf_nbhd_.loc[gdf_nbhd_.std(axis=1) != 0]
# gdf_nbhd_.head()

In [20]:
# meta_col = pd.DataFrame(index=gdf_nbhd_.columns.tolist())
# top_cols = [int(col) for col in gdf_nbhd_.sum(axis=0).sort_values(ascending=False).index]
# predicted_types = pred_cell_types_df.set_index("cluster").loc[top_cols, "category"].tolist()
# meta_col['category'] = predicted_types
# meta_col[:5]

In [21]:
# meta_row = pd.DataFrame(index=gdf_nbhd_.index.tolist())
# top_rows = gdf_nbhd_.sum(axis=1).sort_values(ascending=False).index.tolist()
# niches = gdf_nbhd.set_index("name").loc[top_rows, "niche"].tolist()
# meta_row['niche'] = niches
# meta_row[:5]

In [22]:
# mat = dega.clust.Matrix(
#     gdf_nbhd_,
#     name='parquet',
#     meta_col=meta_col,
#     col_attr=['category'],
#     meta_row=meta_row
# )

# mat.norm(axis='row', by='zscore')
# mat.clust()
# cgm = dega.viz.Clustergram(
#     matrix=mat, 
#     width=500, 
#     height=500
# )
# cgm

## Clustergram: cell_population-by-hextile_nbhd 

In [23]:
gdf_nbhd_ = gdf_nbhd.drop(['geometry', 'leiden', 'niche'], axis=1)
gdf_nbhd_.set_index('name', inplace=True)
gdf_nbhd_ = gdf_nbhd_[(gdf_nbhd_ != 0).any(axis=1)]
gdf_nbhd_ = gdf_nbhd_.loc[gdf_nbhd_.std(axis=1) != 0]
gdf_nbhd_.head()

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
name,,,,,,,,,,,,,,,,,,,,,
hex_25,0.000000,0.191055,0.0,0.219629,0.351398,0.034486,0.017392,0.0,0.000000,0.051293,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hex_26,0.000000,0.167054,0.0,0.110542,0.484508,0.000000,0.050644,0.0,0.012903,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hex_27,0.000000,0.158224,0.0,0.133531,0.424334,0.041964,0.041964,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hex_28,0.020619,0.223144,0.0,0.099091,0.384412,0.010363,0.040822,0.0,0.010363,0.010363,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hex_29,0.154151,0.112478,0.0,0.287682,0.000000,0.000000,0.133531,0.0,0.068993,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
gdf_nbhd_T = gdf_nbhd_.T
gdf_nbhd_T.head()

name,hex_25,hex_26,hex_27,hex_28,hex_29,hex_30,hex_31,hex_32,hex_33,hex_34,...,hex_5772,hex_5801,hex_5803,hex_5804,hex_5805,hex_5806,hex_5807,hex_5808,hex_5809,hex_5810
0,0.000000,0.000000,0.000000,0.020619,0.154151,0.051293,0.225550,0.141651,0.058841,0.071826,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.191055,0.167054,0.158224,0.223144,0.112478,0.219629,0.235120,0.322399,0.200671,0.129534,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.219629,0.110542,0.133531,0.099091,0.287682,0.115832,0.069796,0.063179,0.068319,0.285019,...,0.0,0.000000,0.000000,0.271934,0.259511,0.066691,0.054067,0.405465,0.175891,0.133531
4,0.351398,0.484508,0.424334,0.384412,0.000000,0.084083,0.080969,0.083382,0.241162,0.138836,...,0.0,0.405465,0.502092,0.000000,0.138150,0.033902,0.200671,0.095310,0.207639,0.133531


In [25]:
meta_col = pd.DataFrame(index=gdf_nbhd_T.columns.tolist())
top_cols = gdf_nbhd_T.sum(axis=0).sort_values(ascending=False).index.tolist()
niches = gdf_nbhd.set_index("name").loc[top_cols, "niche"].tolist()
meta_col['niche'] = niches
meta_col[:5]

,niche
hex_25,15
hex_26,5
hex_27,15
hex_28,15
hex_29,1


In [26]:
meta_row = pd.DataFrame(index=gdf_nbhd_T.index.tolist())
top_rows = [int(row) for row in gdf_nbhd_T.sum(axis=1).sort_values(ascending=False).index]
predicted_types = pred_cell_types_df.set_index("cluster").loc[top_rows, "category"].tolist()
meta_row['category'] = predicted_types
meta_row[:5]

,category
0,Smooth muscle
1,Unknown
2,Unknown
3,Endothelial
4,Epithelial


In [27]:
# # Transpose neighborhood matrix
# gdf_nbhd_T = gdf_nbhd_.T

# # Rename index and column names
# gdf_nbhd_T.index.name = "cluster"
# gdf_nbhd_T.columns.name = ""

# # # Ensure cluster index is integer
# # gdf_nbhd_T.index = gdf_nbhd_T.index.astype(int)

# # # Create cluster → cell type mapping
# # cluster_to_cell_type = pred_cell_types_df.set_index("cluster")["predicted_cell_type"].to_dict()

# # # Map cluster index to predicted cell type
# # gdf_nbhd_T.index = gdf_nbhd_T.index.map(cluster_to_cell_type)

# # # Check for unmapped values
# # if gdf_nbhd_T.index.isnull().any():
# #     raise ValueError("Some clusters could not be mapped to predicted cell types.")

# # Build metadata for rows
# top_rows = gdf_nbhd_T.sum(axis=1).sort_values(ascending=False).index
# predicted_types_df = pred_cell_types_df.set_index("predicted_cell_type")

# # Filter and align with top rows (cell types)
# predicted_types = predicted_types_df.loc[top_rows, "category"]
# meta_row = pd.DataFrame(index=top_rows)
# meta_row['category'] = predicted_types.values

# # Build metadata for columns
# top_cols = gdf_nbhd_T.sum(axis=0).sort_values(ascending=False).index
# niches_df = gdf_nbhd.set_index("name")

# # Filter and align with top columns (neighborhoods)
# niches = niches_df.loc[top_cols, "niche"]
# meta_col = pd.DataFrame(index=top_cols)
# meta_col['niche'] = niches.values

In [28]:
mat = dega.clust.Matrix(
    gdf_nbhd_T,
    name='parquet',
    meta_col=meta_col,
    col_attr=['niche'],
    meta_row=meta_row
)

mat.downsample_to(axis='col', category='niche')
mat.norm(axis='row', by='zscore')
mat.clust()
cgm = dega.viz.Clustergram(
    matrix=mat, 
    width=500, 
    height=500,
    entity="NBHD"
)
cgm

Clustergram(entity='NBHD', height=500, network_meta={'linkage': {}, 'row_attr': ['category'], 'col_attr': ['ni…

## Visualize in Landscape view: Hextile NBHD

In [29]:
gdf_nbhd_LF = gdf_nbhd.copy()
gdf_nbhd_LF = gdf_nbhd_LF[['geometry','name','leiden']]
gdf_nbhd_LF.rename(columns={'leiden':'cat'}, inplace=True)
gdf_nbhd_LF.head()

,geometry,name,cat
0,"POLYGON ((6248.35374 58.89656, 6183.40184 96.3...",hex_25,4
1,"POLYGON ((6378.25755 58.89656, 6313.30565 96.3...",hex_26,17
2,"POLYGON ((6508.16136 58.89656, 6443.20946 96.3...",hex_27,17
3,"POLYGON ((6638.06517 58.89656, 6573.11327 96.3...",hex_28,4
4,"POLYGON ((6767.96898 58.89656, 6703.01708 96.3...",hex_29,5


In [30]:
categories = gdf_nbhd_LF['cat'].cat.categories
n_cats = len(categories)

cmap = matplotlib.colormaps['tab20']
colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

cat_to_hex = dict(zip(categories, colors))

gdf_nbhd_LF['color'] = gdf_nbhd_LF['cat'].astype(str).map(cat_to_hex)
gdf_nbhd_LF['area'] = gdf_nbhd_LF['geometry'].area
gdf_nbhd_LF.head()

/var/folders/_6/bhs42vt57t1dkb59k4sy0p440000gp/T/ipykernel_36486/2391661805.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_nbhd_LF['area'] = gdf_nbhd_LF['geometry'].area


,geometry,name,cat,color,area
0,"POLYGON ((6248.35374 58.89656, 6183.40184 96.3...",hex_25,4,#2ca02c,14614.178689
1,"POLYGON ((6378.25755 58.89656, 6313.30565 96.3...",hex_26,17,#dbdb8d,14614.178689
2,"POLYGON ((6508.16136 58.89656, 6443.20946 96.3...",hex_27,17,#dbdb8d,14614.178689
3,"POLYGON ((6638.06517 58.89656, 6573.11327 96.3...",hex_28,4,#2ca02c,14614.178689
4,"POLYGON ((6767.96898 58.89656, 6703.01708 96.3...",hex_29,5,#98df8a,14614.178689


In [31]:
sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
data_dir = f'data/xenium_data/'
path_landscape_files=f'data/landscape_files/{sample}_test'
base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"

# base_url = 'https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_v2/main/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'

landscape_ist = dega.viz.Landscape(
    technology="Xenium",
    base_url = base_url,
    nbhd=gdf_nbhd_LF,
    entity="NBHD"
)

landscape_ist

Landscape(base_url='http://localhost:52973/data/landscape_files/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_te…

In [33]:
dega.viz.landscape_clustergram(landscape_ist, cgm)

## SKIP - Visualize in Landscape view: Niche NBHD

In [27]:
gdf_niche_LF = gdf_niche.copy()
gdf_niche_LF = gdf_niche_LF[['geometry','name','leiden']]
gdf_niche_LF.rename(columns={'leiden':'cat'}, inplace=True)

# Convert 'cat' from categorical strings like '0' → int → +1 → str again
# gdf_niche_LF['cat'] = gdf_niche_LF['cat'].astype(int) + 1
# gdf_niche_LF['cat'] = gdf_niche_LF['cat'].astype(str)
# gdf_niche_LF['cat'] = gdf_niche_LF['cat'].astype('category')

gdf_niche_LF.head()

,geometry,name,cat
0,"MULTIPOLYGON (((6118.44993 3883.89656, 6053.49...",0,0
1,"MULTIPOLYGON (((987.24941 4896.39656, 922.2975...",1,1
2,"MULTIPOLYGON (((272.77845 5158.89656, 272.7784...",2,2
3,"MULTIPOLYGON (((77.92274 3996.39656, 12.97083 ...",3,3
4,"MULTIPOLYGON (((2870.85467 508.89656, 2805.902...",4,4


In [28]:
categories = gdf_niche_LF['cat'].cat.categories
n_cats = len(categories)

cmap = matplotlib.colormaps['tab20']
colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

cat_to_hex = dict(zip(categories, colors))

gdf_niche_LF['color'] = gdf_niche_LF['cat'].astype(str).map(cat_to_hex)
gdf_niche_LF.head()

,geometry,name,cat,color
0,"MULTIPOLYGON (((6118.44993 3883.89656, 6053.49...",0,0,#1f77b4
1,"MULTIPOLYGON (((987.24941 4896.39656, 922.2975...",1,1,#aec7e8
2,"MULTIPOLYGON (((272.77845 5158.89656, 272.7784...",2,2,#ff7f0e
3,"MULTIPOLYGON (((77.92274 3996.39656, 12.97083 ...",3,3,#ffbb78
4,"MULTIPOLYGON (((2870.85467 508.89656, 2805.902...",4,4,#2ca02c


In [29]:
sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
data_dir = f'data/xenium_data/'
path_landscape_files=f'data/landscape_files/{sample}_test'

landscape_ist = dega.viz.Landscape(
    technology="Xenium",
    base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}",
    nbhd=gdf_niche_LF
)

# landscape_ist

## SKIP - Clustergram: niche_nbhd-by-cell_population

In [30]:
gdf_niche_ = gdf_niche.drop(['geometry', 'leiden'], axis=1)
gdf_niche_.set_index('name', inplace=True)

gdf_niche_ = gdf_niche_.apply(pd.to_numeric, errors='coerce')
gdf_niche_ = gdf_niche_.replace([np.inf, -np.inf], np.nan).fillna(0)
gdf_niche_ = gdf_niche_[(gdf_niche_ != 0).any(axis=1)]
gdf_niche_ = gdf_niche_.loc[gdf_niche_.std(axis=1) != 0]
gdf_niche_ = gdf_niche_.loc[:, gdf_niche_.std(axis=0) != 0]

assert np.isfinite(gdf_niche_.values).all(), "Matrix still contains non-finite values!"

In [42]:
meta_col = pd.DataFrame(index=gdf_niche_.columns[1:].tolist())
top_cols = [int(col) for col in gdf_niche_.sum(axis=0).sort_values(ascending=False).index[1:]]
predicted_types = pred_cell_types_df.set_index("cluster").loc[top_cols, "category"].tolist()
meta_col['category'] = predicted_types
meta_col[:5]

,category
0,Epithelial
1,Unknown
2,Fibroblast
3,Macrophage
4,Smooth muscle


In [45]:
gdf_niche_.set_index('niche', inplace=True)

In [47]:
gdf_niche_

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
niche,,,,,,,,,,,,,,,,,,,,,
0,0.000000,0.000000,0.514099,0.000000,0.000000,0.123233,0.063513,0.016261,0.016261,0.000000,...,0.016261,0.000000,0.000000,0.000000,0.000000,0.016261,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.272644,0.014815,0.000000,0.164755,0.043803,0.164755,0.112795,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.014815,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.187212,0.111226,0.257829,0.000000,0.111226,...,0.057158,0.000000,0.111226,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.117783,0.000000,0.000000,0.318454,0.000000,0.117783,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.117783,0.000000
4,0.000000,0.191055,0.000000,0.219629,0.351398,0.034486,0.017392,0.000000,0.000000,0.051293,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.154151,0.112478,0.000000,0.287682,0.000000,0.000000,0.133531,0.000000,0.068993,0.000000,...,0.000000,0.023530,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.125163,0.000000,0.064539,0.182322,0.064539,0.287682,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.064539,0.000000,0.000000,0.064539,0.000000,0.000000
7,0.000000,0.000000,0.312375,0.000000,0.016529,0.371564,0.016529,0.080043,0.048790,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.016529,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,0.074108,0.074108,0.379490,0.207639,0.000000,0.074108,0.000000,...,0.074108,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [48]:
mat = dega.clust.Matrix(
    gdf_niche_,
    name='parquet',
    meta_col=meta_col,
    col_attr=['category'],
)

mat.clust()
cgm = dega.viz.Clustergram(
    matrix=mat, 
    width=500, 
    height=500
)
cgm

Clustergram(height=500, network_meta={'linkage': {}, 'row_attr': [], 'col_attr': ['category'], 'row_attr_maxab…